# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR<sup>2</sup> Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their fields using @id

record_sets = meta.record_set
if not record_sets:
    print("No 'recordSet' found in metadata; attempting to auto-detect via data distributions...")
    # Try getting recordsets from dataset
    record_sets = dataset.get_record_sets()
    if not record_sets:
        raise ValueError("No record sets found in this dataset.")
    else:
        print(f"Detected record sets: {[r['@id'] for r in record_sets]}")
else:
    print(f"Found record sets: {[getattr(r, '@id', None) for r in record_sets]}")

# Print fields (columns) for each record set
rs_ids = []
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None)
    rs_ids.append(rs_id)
    print(f"\nRecord set @id: {rs_id}")
    fields = rs['field'] if 'field' in rs else rs.get('fields') if isinstance(rs, dict) else getattr(rs, 'field', [])
    if not fields:
        print("  No fields defined, will discover from records.")
    else:
        if isinstance(fields, dict):
            fields = [fields]
        for fld in fields:
            field_id = fld['@id'] if isinstance(fld, dict) and '@id' in fld else getattr(fld, '@id', None)
            print(f"  Field @id: {field_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# If no explicit record sets, get from dataset
if len(rs_ids) == 0:
    record_sets_meta = dataset.get_record_sets()
    rs_ids = [r['@id'] for r in record_sets_meta]

print(f"Using record sets: {rs_ids}")

# Extract data from each record set
dataframes = {}
for rs_id in rs_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for {rs_id} with shape {df.shape}")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

# Choose the main record set for further exploration
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"Main record set ID for exploration: {main_rs_id}")
    print("Columns available:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    raise RuntimeError("No dataframes could be loaded. Check source or Croissant schema.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records, normalizing numeric fields, and categorizing data. Adjust field IDs based on the DataFrame above.

In [ ]:
# Select a numeric field for analysis (e.g. 'Molecular Weight' or similar, please specify the correct field ID)
df = dataframes[main_rs_id]

# Attempt to auto-select a numeric field
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if not numeric_cols:
    print("No numeric columns detected. Check DataFrame for available fields.")
    print("Columns: ", df.columns.tolist())
else:
    numeric_field = numeric_cols[0]
    print(f"Selected numeric field: {numeric_field}")

    # Filtering
    threshold = df[numeric_field].quantile(0.75)
    filtered_df = df[df[numeric_field] > threshold].copy()
    n_filtered = len(filtered_df)
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: {n_filtered} rows")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a string/categorical field if present
    group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        display(grouped_df.head())
    else:
        print("No grouping field available.")

## 5. Visualization
Visualize data distributions or relationships between fields using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field histogram
if 'numeric_field' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='steelblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouping is possible, plot grouped means
    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        grouped_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        grouped_means.plot(kind='bar', color='darkorchid')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded FAIR<sup>2</sup> mass spectrometry dataset metadata and records.
- Explored record sets and their available fields using `@id`.
- Loaded main record set into a DataFrame, filtered data by a numeric feature and normalized it.
- Visualized distributions for quick insights.

__You can now proceed with deeper domain-specific analysis as needed by your project!__